In [5]:
import os
import pandas as pd
import numpy as np

# DAVIS 외부검증 결과를 불러온다. qsar_results/dta_results 모두 davis_selectivity.csv를
# 외부 검증셋으로 계산한 {model}/{dataset}/{target}/davis_metrics.csv로 파일명이 통일되어 있어
# 하나의 로더로 두 소스를 동일하게 처리한다.
qsar_base_path = '/data/rlawlsgurjh/work/gitlab/CUE/qsar_results'
dta_base_path = '/data/rlawlsgurjh/work/gitlab/CUE/dta_results'
datasets = ['Kd']  # qsar_results는 현재 Kd 기준으로만 생성되어 있음

# 오직 regression만 사용
label_map = {
    'regression': ['PI', "Ssel", "WS(alpha=2)", "RS(k=2)"]
}
regression_candidates = ['rmse', 'r2', 'pcc']

# 표시 이름 -> (base_dir, base_dir 하위 실제 폴더명)
davis_model_sources = {
    'CUE': (qsar_base_path, 'CMFE'),
    'ADMET-Print': (qsar_base_path, 'ADMET-Print'),
    'ADMET-AI': (qsar_base_path, 'ADMET-AI'),
    'MMFDL': (qsar_base_path, 'MMFDL'),
    'MMFRL': (qsar_base_path, 'MMFRL'),
    'DeepDTA': (dta_base_path, 'DeepDTA'),
    'GraphDTA': (dta_base_path, 'GraphDTA'),
    'TEFDTA': (dta_base_path, 'TEFDTA'),
    'HMM-DTA': (dta_base_path, 'HMM-DTA'),
}

In [6]:
def load_davis_metric(base_dir, model_dir, dataset, target):
    """{base_dir}/{model_dir}/{dataset}/{target}/davis_metrics.csv (DAVIS 외부검증, 단일 행)에서
    mean/std 값을 읽어온다. qsar_results/dta_results 모두 동일 파일명(davis_metrics.csv)으로
    통일되어 있어 분기 없이 하나의 로더로 처리한다."""
    metrics_path = os.path.join(base_dir, model_dir, dataset, target, 'davis_metrics.csv')
    if not os.path.exists(metrics_path):
        return None
    try:
        df = pd.read_csv(metrics_path)
        if df.empty:
            return None
        row = df.iloc[0]
        res = {}
        for m in regression_candidates:
            if m in row.index:
                res[m] = row[m]
                res[f'{m}_std'] = row.get(f'{m}_std', np.nan)
        return res if res else None
    except Exception:
        return None


raw_data_list = []
for d_name in datasets:
    for target in label_map['regression']:
        for display_name, (base_dir, model_dir) in davis_model_sources.items():
            m = load_davis_metric(base_dir, model_dir, d_name, target)
            if m:
                entry = {'Dataset_Group': d_name, 'Target': target, 'Modality': display_name}
                entry.update(m)
                raw_data_list.append(entry)

df_all = pd.DataFrame(raw_data_list)

In [7]:
df_all.head(10)

,Dataset_Group,Target,Modality,rmse,rmse_std,r2,r2_std,pcc,pcc_std
0,Kd,PI,CUE,0.280637,0.042945,-0.326829,0.414508,0.561175,0.113693
1,Kd,PI,ADMET-Print,0.272020,0.016446,-0.227249,0.146045,0.394211,0.034171
2,Kd,PI,ADMET-AI,0.369060,0.034506,-1.268214,0.424921,0.213842,0.100853
3,Kd,PI,MMFDL,0.390147,0.034666,-1.533110,0.458653,0.082314,0.094032
4,Kd,PI,MMFRL,0.222128,0.033192,0.169465,0.227742,0.634835,0.106167
5,Kd,PI,DeepDTA,0.363090,0.003698,-1.180358,0.044205,0.556846,0.002289
6,Kd,PI,GraphDTA,0.349806,0.021867,-1.029893,0.247560,0.549098,0.007224
7,Kd,PI,TEFDTA,0.356741,0.012122,-1.106547,0.144775,0.543485,0.006900
8,Kd,PI,HMM-DTA,0.277338,0.007635,-0.272762,0.070012,0.399480,0.095663
9,Kd,Ssel,CUE,1.166640,0.147453,-0.357969,0.338394,0.486927,0.072101


In [8]:
# 지표별 방향 표시 맵
metric_direction = {
    'rmse': '↓', 'r2': '↑', 'pcc': '↑'
}

save_base_dir = os.path.join('/data/rlawlsgurjh/work/gitlab/CUE/cue/results', 'table', 'davis_comparison')
os.makedirs(save_base_dir, exist_ok=True)

# DTA 베이스라인(내부 CV) + DAVIS 외부검증 모델(CUE 포함)
model_order = ['DeepDTA', 'GraphDTA', 'TEFDTA', 'HMM-DTA', 'ADMET-Print', 'ADMET-AI', 'MMFDL', 'MMFRL', 'CUE']

for d_name in datasets:
    df_subset = df_all[df_all['Dataset_Group'] == d_name]
    
    metrics = regression_candidates
    targets = label_map['regression']
    
    task_table_rows = []
    for target in targets:
        for model in model_order:
            m_data = df_subset[(df_subset['Target'] == target) & (df_subset['Modality'] == model)]
            row = {'Target': target, 'Model': model}
            
            for m in metrics:
                m_header = f"{m.upper()}( {metric_direction.get(m, '')} )"
                
                if not m_data.empty and m in m_data.columns:
                    mean_val = m_data[m].values[0]
                    std_val = m_data[f"{m}_std"].values[0] if f"{m}_std" in m_data.columns else 0.0
                    # mean ± std (셋째 자리까지)
                    row[m_header] = f"{mean_val:.3f} ± {std_val:.3f}"
                else:
                    row[m_header] = "N/A"
            
            # CUE이거나, 유효한 값이 하나라도 있으면 저장
            if model == 'CUE' or any(row[f"{m.upper()}( {metric_direction.get(m, '')} )"] != "N/A" for m in metrics):
                task_table_rows.append(row)
        
    g_name = 'DIS' if d_name == 'Kd' else 'INH'

    if task_table_rows:
        res_df = pd.DataFrame(task_table_rows)
        save_path = os.path.join(save_base_dir, f"{g_name}_results.csv")
        res_df.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
        print(f"📄 통합 결과 저장 완료: {save_path}")

display(res_df)

📄 통합 결과 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/table/davis_comparison/DIS_results.csv


,Target,Model,RMSE( ↓ ),R2( ↑ ),PCC( ↑ )
0,PI,DeepDTA,0.363 ± 0.004,-1.180 ± 0.044,0.557 ± 0.002
1,PI,GraphDTA,0.350 ± 0.022,-1.030 ± 0.248,0.549 ± 0.007
2,PI,TEFDTA,0.357 ± 0.012,-1.107 ± 0.145,0.543 ± 0.007
3,PI,HMM-DTA,0.277 ± 0.008,-0.273 ± 0.070,0.399 ± 0.096
4,PI,ADMET-Print,0.272 ± 0.016,-0.227 ± 0.146,0.394 ± 0.034
5,PI,ADMET-AI,0.369 ± 0.035,-1.268 ± 0.425,0.214 ± 0.101
6,PI,MMFDL,0.390 ± 0.035,-1.533 ± 0.459,0.082 ± 0.094
7,PI,MMFRL,0.222 ± 0.033,0.169 ± 0.228,0.635 ± 0.106
8,PI,CUE,0.281 ± 0.043,-0.327 ± 0.415,0.561 ± 0.114
9,Ssel,DeepDTA,2.141 ± 0.029,-3.515 ± 0.120,0.707 ± 0.000
